In [ ]:
from langchain_core.tools import tool
from langchain.tools import ToolRuntime
import uuid

@tool
def book_ticket(
    tickets: int,
    movie_name: str,
    runtime: ToolRuntime
):
    """
    Book one or more movie tickets.

    Args:
        tickets: Number of tickets to book. Must be between 1 and 10.
        movie_name: Name of the movie.
    """

    user_name = runtime.context.user_name

    # Read existing tickets
    item = runtime.store.get(
        "user-ticket-map",
        user_name
    )

    tickets_list = item.value if item else []

    # Validate
    if tickets <= 0 or tickets > 10:
        return "Please book between 1 and 10 tickets only."

    # Generate ticket IDs
    ticket_ids = [
        str(uuid.uuid4())
        for _ in range(tickets)
    ]

    # Create booking
    ticket_obj = {
        "total_tickets": tickets,
        "ticket_ids": ticket_ids,
        "movie_name": movie_name,
    }

    # Add booking
    tickets_list.append(ticket_obj)

    # Persist
    runtime.store.put(
        "user-ticket-map",
        user_name,
        tickets_list
    )

    return (
        f"Hello {user_name}, {tickets} tickets have been booked "
        f"for {movie_name}. "
        f"Ticket IDs: {ticket_ids}"
    )

In [ ]:
@tool
def book_ticket(request_details: dict, runtime: ToolRuntime):
    """
    This function is used to book a ticket for a movie
    """

    user_name = runtime.context.user_name

    # read existing tickets for the user from the same namespace/key we write to
    item = runtime.store.get("user-ticket-map", user_name)
    tickets = item.value if item else []

    # details given by the user.
    if not isinstance(request_details, dict):
        return "I could not read the ticket request. Please specify the number of tickets and movie name."

    number_of_tickets = int(request_details.get("tickets"))
    movie_name = request_details.get("movie_name")

    if number_of_tickets is None or movie_name is None:
        return "Please provide both the number of tickets and the movie name."

    try:
        number_of_tickets = int(number_of_tickets)
    except (TypeError, ValueError):
        return "The number of tickets must be a valid integer."

    if number_of_tickets <= 0 or number_of_tickets > 10:
        return "Please book between 1 and 10 tickets only."

    # list of newly generated ticket_ids
    list_of_new_ticket_ids = []

    # generating ticket_ids and appending it into the list
    for _ in range(number_of_tickets):
        random_uuid = uuid.uuid4()
        ticket_id = str(random_uuid)
        list_of_new_ticket_ids.append(ticket_id)

    # creating the new ticket object
    ticket_obj = {
        "total_tickets": number_of_tickets,
        "ticket_ids": list_of_new_ticket_ids,
        "movie_name": movie_name,
    }

    # appending the ticket-object into the tickets array
    tickets.append(ticket_obj)

    runtime.store.put("user-ticket-map", user_name, tickets)


    return f"Hello {user_name}, {number_of_tickets} tickets has been booked with following details {tickets}"

In [ ]:
response_three = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Cancel the ticket with ticket id 0fdcd6f6-afc8-4458-b223-f0b379c244e4 for ramayan"}
        ]
    },
    config=config,
    context=Context(
        user_name="uttam kumar",
        email="uttamkumar@gmail.com"
    )
)

# print(response_three)
print(response_three["messages"][-1].content)

In [ ]:
response = agent.invoke(
    Command(resume={"decisions":[{"type":"approve"}]}),
    config=config
)
print(response)
print(response["messages"][-1].content)